In [11]:
%pip install xgboost
# train_and_save_model.py
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
import xgboost as xgb
import warnings
import os

warnings.filterwarnings('ignore', category=UserWarning)

def train_and_save_models():
    """
    Loads data, trains multiple models, selects the best one,
    and saves the model, scaler, and label encoder to disk.
    """
    print("Starting model training and saving process...")
    # Load and preprocess data with error handling
    try:
        data_file = "ADXL345_SensorData.csv"
        if not os.path.exists(data_file):
            raise FileNotFoundError(f"Error: '{data_file}' not found. Please provide the correct file path.")
        
        df = pd.read_csv(data_file)
        if df.empty or 'Error_found' not in df.columns or df[['X-direction', 'Y-direction', 'Z-direction']].isnull().all().any():
            raise ValueError("Dataset is empty or missing required columns (X-direction, Y-direction, Z-direction, Error_found)")
    except Exception as e:
        print(f"Error loading dataset: {str(e)}")
        return

    # Encode target variable
    le = LabelEncoder()
    # Ensure 'yes'/'no' mapping is handled correctly before encoding
    df['Error_found'] = df['Error_found'].astype(str).replace({'yes': 1, 'no': 0})
    df['Error_found'] = le.fit_transform(df['Error_found']) # 'no' -> 0, 'yes' -> 1
    
    print("Data loaded and preprocessed successfully.")

    # Feature and target split
    X = df[['X-direction', 'Y-direction', 'Z-direction']]
    y = df['Error_found']

    # Feature scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print("Data scaled.")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # Define models with probability support
    models = {
        'Logistic Regression': LogisticRegression(),
        'Naive Bayes': GaussianNB(),
        'SVM': SVC(probability=True),
        'KNN': KNeighborsClassifier(),
        'Decision Tree': DecisionTreeClassifier(),
        'Random Forest': RandomForestClassifier(),
        'Gradient Boosting': GradientBoostingClassifier(),
        'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    }

    results = {}
    
    # Train and evaluate models
    print("Training and evaluating models...")
    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            results[name] = acc
            print(f"  - {name}: Accuracy = {acc:.4f}")
        except Exception as e:
            results[name] = 0.0
            print(f"  - Error training {name}: {str(e)}")

    # Train and evaluate Voting Ensemble
    try:
        estimators = [(name, model) for name, model in models.items()]
        voting_model = VotingClassifier(estimators=estimators, voting='soft')
        voting_model.fit(X_train, y_train)
        y_pred_voting = voting_model.predict(X_test)
        results['Voting Ensemble'] = accuracy_score(y_test, y_pred_voting)
        print(f"  - Voting Ensemble: Accuracy = {results['Voting Ensemble']:.4f}")
    except Exception as e:
        results['Voting Ensemble'] = 0.0
        print(f"  - Error training Voting Ensemble: {str(e)}")

    # Identify best model
    best_model_name = max(results, key=results.get)
    best_model = voting_model if best_model_name == 'Voting Ensemble' else models[best_model_name]

    print(f"\nBest performing model is: {best_model_name} with accuracy {results[best_model_name]:.4f}")

    # Save the best model, scaler, and label encoder
    try:
        joblib.dump(best_model, 'best_model.pkl')
        joblib.dump(le, 'label_encoder.pkl')
        joblib.dump(scaler, 'scaler.pkl')
        print("Model, scaler, and label encoder saved successfully.")
    except Exception as e:
        print(f"Error saving files: {str(e)}")

if __name__ == '__main__':
    train_and_save_models()

Note: you may need to restart the kernel to use updated packages.
Starting model training and saving process...
Data loaded and preprocessed successfully.
Data scaled.
Training and evaluating models...
  - Logistic Regression: Accuracy = 1.0000
  - Naive Bayes: Accuracy = 1.0000
  - SVM: Accuracy = 1.0000
  - KNN: Accuracy = 1.0000
  - Decision Tree: Accuracy = 1.0000
  - Random Forest: Accuracy = 1.0000
  - Gradient Boosting: Accuracy = 1.0000
  - XGBoost: Accuracy = 1.0000
  - Voting Ensemble: Accuracy = 1.0000

Best performing model is: Logistic Regression with accuracy 1.0000
Model, scaler, and label encoder saved successfully.
